# Redox FHIR Pipeline - Resources Exploded

This notebook explodes FHIR bundle entries into individual resources and infers their schemas for dynamic Silver table creation.

## Tables Created
| Table | Description |
|-------|-------------|
| `resources_exploded` | Flattened key-value pairs for each resource |
| `resource_schemas` | Inferred schemas per resource type |

## Processing Flow
```
fhir_bronze_variant.entry[] → LATERAL variant_explode → resources_exploded
                                                            ↓
                                            schema_of_variant_agg → resource_schemas
```

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE schema_use STRING DEFAULT 'bronze';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE schema_use = COALESCE(:schema_use, schema_use);

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog() AS catalog, current_schema() AS schema;

## Step 1: Explode Bundle Entries into Key-Value Pairs

Each FHIR bundle contains an `entry` array with resources. This step:
1. Explodes the `entry` array to get individual resources
2. Explodes each resource's fields into key-value pairs
3. Preserves `bundle_uuid` and `fullUrl` for lineage

In [ ]:
CREATE OR REFRESH STREAMING TABLE resources_exploded (
  bundle_uuid STRING NOT NULL COMMENT 'Reference to parent bundle'
  ,full_url STRING COMMENT 'FHIR fullUrl for the resource'
  ,resource_type STRING COMMENT 'FHIR resource type (Patient, Claim, etc.)'
  ,key STRING COMMENT 'Resource field name'
  ,value VARIANT COMMENT 'Resource field value as VARIANT'
)
COMMENT 'Exploded FHIR resources with key-value pairs'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'bronze'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS SELECT
  bundle_uuid
  ,CAST(entry.value:fullUrl AS STRING) AS full_url
  ,CAST(entry.value:resource.resourceType AS STRING) AS resource_type
  ,resource.key AS key
  ,resource.value AS value
FROM
  STREAM(fhir_bronze_variant)
  ,LATERAL variant_explode(fhir:entry) AS entry
  ,LATERAL variant_explode(entry.value:resource) AS resource;

In [ ]:
-- Verify exploded resources
SELECT 
  resource_type,
  key,
  value,
  bundle_uuid
FROM resources_exploded
WHERE resource_type = 'Patient'
LIMIT 20;

In [ ]:
-- Count resources by type
SELECT 
  resource_type,
  COUNT(DISTINCT bundle_uuid || full_url) AS resource_count,
  COUNT(DISTINCT key) AS unique_fields
FROM resources_exploded
GROUP BY resource_type
ORDER BY resource_count DESC;

## Step 2: Infer Schemas from VARIANT Data

This step analyzes all values for each resource type and field to infer the schema. This enables dynamic Silver table creation without predefined schemas.

In [ ]:
CREATE OR REFRESH STREAMING TABLE resource_schemas (
  resource_type STRING COMMENT 'FHIR resource type'
  ,column_name STRING COMMENT 'Field name (will become column in Silver)'
  ,schema_of_variant STRING COMMENT 'Inferred VARIANT schema'
  ,schema_as_struct STRING COMMENT 'Schema converted to STRUCT notation'
)
COMMENT 'Inferred schemas for each FHIR resource type'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'bronze'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS SELECT
  resource_type
  ,key AS column_name
  ,schema_of_variant_agg(value) AS schema_of_variant
  ,REPLACE(schema_of_variant_agg(value), 'OBJECT', 'STRUCT') AS schema_as_struct
FROM STREAM(resources_exploded)
GROUP BY
  resource_type
  ,key;

In [ ]:
-- View inferred schemas
SELECT 
  resource_type,
  column_name,
  schema_of_variant
FROM resource_schemas
ORDER BY resource_type, column_name
LIMIT 50;

In [ ]:
-- List all discovered resource types
SELECT DISTINCT resource_type
FROM resource_schemas
ORDER BY resource_type;

In [ ]:
-- Schema summary per resource type
SELECT 
  resource_type,
  COUNT(*) AS field_count,
  COLLECT_LIST(column_name) AS fields
FROM resource_schemas
GROUP BY resource_type
ORDER BY field_count DESC;